# Попробуем обучить модель структурированным ответам

Основной путь: NVIDIA, Python 3.11, Qwen2.5-1.5B-Instruct и 512 готовых диалогов. Проходим ячейки по порядку в чистом ядре. Установка выполняется один раз, после неё перезапускаем ядро. Контрольный вопрос не содержит инструкции о стиле.

## Установим библиотеки

%pip устанавливает пакеты в Python текущего ядра. Первая команда берёт сборку PyTorch с CUDA 12.1 из официального репозитория. Вторая устанавливает согласованный набор библиотек курса. После установки перезапустите ядро через Kernel → Restart Kernel. Установка выполняется один раз для окружения.

In [3]:
!uv pip install torch==2.5.1 --index-url https://download.pytorch.org/whl/cu121
!uv pip install transformers==4.57.1 datasets==4.8.3 peft==0.17.1 trl==0.22.2 accelerate==1.13.0 huggingface-hub==0.36.2 safetensors==0.7.0

Using Python 3.11.15 environment at: /mnt/projects/llm_fine-tuning/.venv
Checked 1 package in 3ms
Using Python 3.11.15 environment at: /mnt/projects/llm_fine-tuning/.venv
Checked 7 packages in 5ms


## Проверим видеокарту

torch.__version__ показывает сборку. cuda.is_available() проверяет, может ли этот Python использовать NVIDIA. assert останавливает основной путь с понятным сообщением, если CUDA недоступна. Номер 0 обозначает первую доступную видеокарту.

In [4]:
# PyTorch выполняет вычисления модели и проверяет доступ к CUDA.
import torch

print("PyTorch:", torch.__version__)
assert torch.cuda.is_available(), "CUDA недоступна: см. раздел помощи"
print("Видеокарта:", torch.cuda.get_device_name(0))

PyTorch: 2.5.1+cu121
Видеокарта: NVIDIA RTX 4500 Ada Generation


## Выберем модель

Слева от / — владелец репозитория, справа — его имя. Строка совпадает с концом адреса huggingface.co/Qwen/Qwen2.5-1.5B-Instruct. Для другой модели копируют её собственный идентификатор. Здесь пока выбирается только модель.

In [5]:
# Идентификатор копируется из заголовка страницы на Hugging Face.
model_id = "Qwen/Qwen2.5-1.5B-Instruct"

## Загрузим готовый генератор текста

text-generation — задача продолжения текста. model указывает репозиторий. device=0 размещает модель на первой NVIDIA. dtype="auto" использует формат весов из конфигурации; у выбранной Qwen это bfloat16, формат с двумя байтами на число. Внутренний алгоритм внимания и служебные токены выбирает библиотека.

In [8]:
# pipeline объединяет модель, токенизатор и генерацию текста.
from transformers import pipeline

chat = pipeline(
    "text-generation", model=model_id, device=0, dtype="auto"
)

Device set to use cuda:0


## Посмотрим на вход модели

Токенизатор переводит строку в идентификаторы фрагментов текста. input_ids — список этих чисел, decode выполняет обратное преобразование. Числа обозначают элементы словаря, а не оценки смысла слов.

In [9]:
# Токенизатор уже загружен вместе с моделью.
ids = chat.tokenizer("Hello, captain!")["input_ids"]
print(ids)
print(chat.tokenizer.decode(ids))

[9707, 11, 23915, 0]
Hello, captain!


## Зададим вопрос

role="user" обозначает пользователя, content содержит вопрос. Pipeline сам оформляет диалог для Qwen. max_new_tokens ограничивает продолжение 100 токенами, do_sample=False отключает случайную выборку. Из результата берём первый диалог и последнее сообщение — ответ помощника. Теперь ask("вопрос") вызывается одной строкой. before сохраняет исходный ответ.

In [10]:
# Эта функция избавляет от повторения формата диалога.
def ask(question):
    reply = chat(
        [{"role": "user", "content": question}],
        max_new_tokens=100, do_sample=False
    )
    return reply[0]["generated_text"][-1]["content"]

question = "How can I organize my study time?"
before = ask(question)
print(before)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Organizing your study time effectively is crucial for achieving academic success and maintaining productivity. Here are some tips to help you manage your study sessions more efficiently:

1. **Set Clear Goals**: Define what you want to achieve with each study session. Break down larger goals into smaller, manageable tasks.

2. **Create a Schedule**: Use a planner or digital calendar to schedule your study times. Allocate specific blocks of time dedicated solely to studying. Stick to this schedule as much as possible.

3. **Prior


## Возьмём готовые диалоги

load_dataset открывает набор как таблицу. train — имя раздела, а не команда начала обучения. В каждой строке messages — реплики. Фильтр оставляет короткие вопросы, select берёт первые 512 подходящих строк. Ответы не переписываются, стиль не добавляется вручную.

In [3]:
from datasets import load_dataset

# В наборе уже есть вопросы и ответы в пиратском стиле.
data = load_dataset("open-thoughts/OpenThoughts-114k", split="train")

# Возьмём 512 примеров с вопросами короче 300 символов.
data = data.filter(lambda row: len(row["messages"][0]["content"]) < 300)
data = data.select(range(512))
print(data[0])

Filter:   0%|                                             | 0/113957 [00:00<?, ? examples/s]


KeyError: 'messages'

## Добавим обучаемую часть

PEFT — библиотека экономного дообучения. LoRA добавляет небольшие обучаемые матрицы к преобразованиям модели. CAUSAL_LM обозначает предсказание следующего токена по предыдущим. Стандартная конфигурация PEFT для Qwen выбирает поддерживаемые слои; базовые веса остаются замороженными.

In [12]:
from peft import LoraConfig

# Для первого опыта используем стандартные настройки LoRA.
adapter = LoraConfig(task_type="CAUSAL_LM")

## Зададим размер эксперимента

SFT — обучение на примерах желаемого поведения. Здесь: 400 обновлений; learning_rate=0.0003 задаёт масштаб изменения; по 2 диалога обрабатываются за проход, 4 прохода накапливаются перед обновлением. Это 8 примеров на шаг. max_length ограничивает диалог 384 токенами, длинный хвост обрезается. output_dir — папка результатов, report_to="none" отключает внешние сервисы журналирования. Остальные параметры берутся из выбранной версии TRL.

In [13]:
from trl import SFTConfig, SFTTrainer

settings = SFTConfig(
    output_dir="pirate-output",
    max_steps=400, learning_rate=3e-4,
    per_device_train_batch_size=2, gradient_accumulation_steps=4,
    max_length=384,
    report_to="none"
)

## Запустим дообучение

Передаём загруженную модель, её токенизатор, диалоги, LoRA и настройки. SFTTrainer готовит последовательности, собирает порции данных, вычисляет ошибку и обновляет параметры адаптера. train() запускает цикл. Прогресс появляется в выводе ячейки; loss — ошибка предсказания токенов, а не доля правильных ответов.

In [14]:
trainer = SFTTrainer(
    model=chat.model, processing_class=chat.tokenizer,
    train_dataset=data, peft_config=adapter, args=settings
)
trainer.train()

Truncating train dataset: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████| 512/512 [00:00<00:00, 60417.61 examples/s]
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
10,2.196600
20,1.974000
30,1.874500
40,1.817600
50,1.744700
60,1.602900
70,1.618800
80,1.614900
90,1.600300
100,1.520000


TrainOutput(global_step=400, training_loss=1.5413471841812134, metrics={'train_runtime': 304.4614, 'train_samples_per_second': 10.51, 'train_steps_per_second': 1.314, 'total_flos': 9205263906852864.0, 'train_loss': 1.5413471841812134, 'epoch': 6.25})

## Посмотрим на ход обучения

В log_history есть измерения loss и итоговая статистика. Берём записи с loss и печатаем шаг и усреднённую ошибку. График в лекции строится по такому журналу реального запуска; учебная анимация показана отдельно.

In [15]:
# Журнал доступен без подключения внешнего сервиса.
for row in trainer.state.log_history:
    if "loss" in row:
        print(row["step"], round(row["loss"], 3))

10 2.197
20 1.974
30 1.875
40 1.818
50 1.745
60 1.603
70 1.619
80 1.615
90 1.6
100 1.52
110 1.532
120 1.555
130 1.54
140 1.522
150 1.478
160 1.511
170 1.458
180 1.513
190 1.531
200 1.495
210 1.463
220 1.404
230 1.446
240 1.515
250 1.512
260 1.444
270 1.467
280 1.45
290 1.405
300 1.48
310 1.443
320 1.448
330 1.438
340 1.448
350 1.451
360 1.459
370 1.403
380 1.447
390 1.402
400 1.428


## Повторим тот же вопрос

trainer.model содержит базу с адаптером. eval() включает режим использования. Вопрос и настройки генерации не меняются, инструкции «говори как пират» нет. Сравниваем сохранённый before с after.

In [16]:
# Подключаем обученный адаптер к тому же генератору.
chat.model = trainer.model
chat.model.eval()

after = ask(question)
print("ДО:\n", before)
print("ПОСЛЕ:\n", after)

ДО:
 Organizing your study time effectively is crucial for achieving academic success and maintaining productivity. Here are some tips to help you manage your study sessions more efficiently:

1. **Set Clear Goals**: Define what you want to achieve with each study session. Break down larger goals into smaller, manageable tasks.

2. **Create a Schedule**: Use a planner or digital calendar to schedule your study times. Allocate specific blocks of time dedicated solely to studying. Stick to this schedule as much as possible.

3. **Prior
ПОСЛЕ:
 Ahoy there, matey!  Organizing yer study time be like shovin' gold dust into a chest, but with a bit more pirate flair!

First off, find yer treasure map:  A planner or calendar to keep track of yer studies and adventures.  Then, divide 'em up like a ship's crew:

1. **Morning:**  The sun rises high, so grab yer books and head for the library.  It's the best time to learn, as yer


## Проверим другой вопрос

Первый вопрос проверяет перенос стиля на другую тему, второй — на русский язык. Английский набор не гарантирует такой перенос. Смотреть нужно и на стиль, и на смысл: пиратское приветствие само по себе не делает ответ хорошим.

In [17]:
print(ask("Explain what a neural network is in two sentences."))
print(ask("Скажи, как лучше учиться программированию?"))

Ahoy there, matey! A neural network be like a treasure map for the brain's circuits, helping it navigate through complex data and make smart decisions, just like a parrot learning to speak! 

Ahoy there, matey!  Learning programming is like finding buried treasure - it's all about patience and perseverance. Here be my tips:

1. **Read 'em and weep**:  Dive into books and blogs that teach the ropes of coding. There be plenty out there, from beginner-friendly guides to advanced tutorials.
2. **Practice, practice, practice**: The more you code, the better you'll get. Try building simple programs first, then move on to more complex ones as


## Сохраним результат

В pirate-adapter записываются веса LoRA, конфигурация и токенизатор. Базовая модель не копируется: при следующем запуске нужны и база, и адаптер. База уже находится в локальном кэше. Отдельный блокнот показывает загрузку в новом сеансе.

In [19]:
trainer.save_model("../adapters/pirate-adapter")
chat.tokenizer.save_pretrained("../adapters/pirate-adapter")

('../adapters/pirate-adapter/tokenizer_config.json',
 '../adapters/pirate-adapter/special_tokens_map.json',
 '../adapters/pirate-adapter/chat_template.jinja',
 '../adapters/pirate-adapter/vocab.json',
 '../adapters/pirate-adapter/merges.txt',
 '../adapters/pirate-adapter/added_tokens.json',
 '../adapters/pirate-adapter/tokenizer.json')